# Lecture: Designing an Interactive Dashboard with Plotly Dash

A dashboard brings together a small set of related evidence so that a particular audience can answer a particular question.

We will first run the smallest possible Dash app. Then we will use the reproducible analysis workflow to plan and build a school-age micromobility injury dashboard one piece at a time.


## Learning goals

By the end of this lesson, you should be able to:

- identify the audience, purpose, and central question for a dashboard;
- apply visual hierarchy, consistency, accessibility, and restraint;
- explain the roles of a Dash app, layout, component, and callback;
- connect an input to one or more outputs using component IDs and properties;
- assemble a small interactive dashboard from checked evidence; and
- communicate the source, population, and limitations of dashboard evidence.


# First: run a very small Dash app

Let us see the result before examining all of the pieces. Dash builds interactive web applications containing text, controls, and Plotly figures. We do not need to write JavaScript for this example.

If Dash is not installed, run `%pip install dash` once in a separate cell and restart the kernel if prompted.


In [ ]:
from dash import Dash, html


`Dash(__name__)` creates the application. We store it in `first_app`.


In [ ]:
first_app = Dash(__name__)


Every Dash app needs a **layout** describing what appears on the page. `html.Div` is a container. `html.H1` creates a heading, and `html.P` creates a paragraph. The list inside the `Div` contains its children.


In [ ]:
first_app.layout = html.Div(
    [
        html.H1("My first Dash app"),
        html.P("This page was created with Python."),
    ]
)


Run the next cell to open the application in a browser tab. `jupyter_mode="tab"` tells Dash to open a tab instead of embedding the page inside the notebook. Each app in this lesson uses a different `port` so that its callbacks cannot be confused with callbacks from an earlier app.


In [ ]:
first_app.run(jupyter_mode="tab", port=8050, debug=False)


## What just happened?

1. `Dash(__name__)` created an application.
2. `.layout` described the page.
3. `.run()` started the application.

The app is not interactive yet. Before adding interaction, we need a clear purpose and trustworthy evidence.


# The reproducible analysis workflow

1. **Question:** What do we want to learn?
2. **Data:** Which observations and features can answer the question?
3. **Operation:** What should the code select, calculate, or visualize?
4. **Check:** Does the result have sensible rows, values, units, and missingness?
5. **Evidence:** Which result directly answers the question?
6. **Conclusion:** What claim is supported?
7. **Limitation:** What should we avoid concluding?

A dashboard packages selected operations and evidence into an interface. It does not replace the workflow.


# Step 1 — Question

Orange County Public Schools announced a 2026–2027 decision concerning student use of e-bikes and e-scooters. Elementary and middle-school students may not bring these devices onto school property. High-school students may do so only with a learner's permit or driver's license and an approved school decal.

- [OCPS explanation of the 2026–2027 school-board decision](https://theslice.ocps.net/88316?articleID=72703)

The decision raises a useful descriptive question for a safety researcher:

> **Among patients ages 5–17 in this 2025 dataset, how did emergency-department injury patterns differ by patient age and month between non-electric bicycles and e-bikes, and between unpowered and powered scooters?**

The dashboard will support exploration of age, month, injury outcomes, body parts, and diagnoses. It cannot tell us what effect the OCPS school-board decision will have.


## Principles of good dashboard design

- **Audience and purpose:** Know who will use it and what it should help them understand.
- **Clear focus:** Every element should contribute to the central question.
- **Visual hierarchy:** Put important controls and summaries first.
- **Complementary views:** Each chart should add evidence rather than repeat another chart.
- **Consistency:** Use stable labels, colors, units, and definitions.
- **Accessibility:** Use readable text, strong contrast, and descriptive labels.
- **Purposeful interaction:** Add a control only when it supports a useful follow-up question.
- **Useful default:** The first view should make sense before the user changes anything.
- **Context:** Keep the data source and important limitations near the evidence.

More cards, controls, and charts do not automatically make a better dashboard. They also create more work for the user.


## Our dashboard plan

| Component | What it contributes |
| --- | --- |
| Device-comparison dropdown | Shows device families, power groups, or all four device types |
| Three summary cards | Provide case count, median patient age, and the share needing additional care |
| Age-or-month toggle and line chart | Compares cases by patient age or calendar month for the selected devices |
| Additional-care chart | Compares the share needing additional care |
| Injury-detail controls and bar chart | Compare a selected diagnosis and body part across all four device types |
| Source note | States the population, year, definitions, and limitations |


## Discussion: predict the comparison

1. Do you expect the electric or powered device in each pair to have more cases in the dataset? Why?
2. Do you expect the electric or powered device in each pair to have a larger share of cases needing additional care? Why?
3. Which months do you expect to have the most cases? Why?
4. Which view should receive the most space on the page?


### Discussion notes

Predictions will vary. Students should identify that case counts may reflect both how often each device appears in this hospital sample and its injury patterns. A larger or faster device might plausibly produce different injury outcomes, but the dashboard cannot measure speed, exposure, or causation.


# Step 2 — Data

The U.S. Consumer Product Safety Commission (CPSC) uses the National Electronic Injury Surveillance System (NEISS) to collect a probability sample of product-related injuries treated in U.S. hospital emergency departments.

The class dataset was prepared from the complete result of an official [CPSC NEISS query](https://www.cpsc.gov/cgibin/NEISSQuery/home.aspx) for 2025 cases involving the following product codes:

| Dashboard group | CPSC product code(s) |
| --- | --- |
| Non-electric bicycle | 5033 and 5040 |
| E-bike | 5045 |
| Unpowered scooter | 5023 |
| Powered scooter | 5022 |

CPSC calls code 5022 **powered scooters**. That category includes many e-scooters, but it does not guarantee that every device was electrically powered. We will preserve the official label rather than claim more precision than the data provide.

Before class, the source data were prepared so that we can focus on dashboard design:

- records were limited to patients ages 5–17;
- 97 records containing more than one dashboard group were removed so the groups would not overlap;
- product, sex, race, primary body-part, primary diagnosis, and disposition codes were replaced with readable labels;
- only the first body-part and diagnosis fields were retained; and
- device family, device type, month, and an emergency-department outcome feature were calculated.

The original files also contain `Weight`, `PSU`, and `Stratum`, which CPSC uses when producing national estimates and measuring their uncertainty. We will not make national estimates in this lesson, so those survey-design fields have been omitted. Every number in this dashboard describes the unweighted cases in the prepared class dataset.


In [ ]:
import pandas as pd
import plotly.express as px
from dash import Input, Output, dcc


In [ ]:
injuries = pd.read_csv(
    "data/cpsc_neiss_school_age_micromobility_2025.csv",
    parse_dates=["treatment_date"],
)
injuries.head()


This uses `pd.read_csv()`, as in our previous lessons and labs. `parse_dates=["treatment_date"]` stores the treatment date as a datetime feature.


In [ ]:
injuries.tail()


In [ ]:
injuries.info()


In [ ]:
injuries.isnull().sum()


## Data dictionary for the features we will use

| Feature | What it records |
| --- | --- |
| `case_number` | Identifier for one sampled emergency-department case |
| `treatment_date` | Calendar date on which the patient was treated |
| `age` | Patient age in years |
| `sex` | Patient sex reported in the source record |
| `race` | Patient race reported in the source record |
| `body_part` | Primary body part affected by the injury |
| `diagnosis` | Primary diagnosis recorded for the injury |
| `disposition` | Outcome after emergency-department treatment |
| `device_type` | One of the four mutually exclusive device groups |
| `device_family` | `Bicycles` or `Scooters` |
| `month` | Month of the treatment date |
| `additional_care` | `True` when the case was grouped as needing additional care after emergency-department treatment |


## Discussion: check the dashboard data

Use `.info()` and `.isnull().sum()` to complete the table.

| Feature | Feature type | Data storage type | Missing values |
| --- | --- | --- | ---: |
| `treatment_date` |  |  |  |
| `age` |  |  |  |
| `sex` |  |  |  |
| `race` |  |  |  |
| `body_part` |  |  |  |
| `diagnosis` |  |  |  |
| `disposition` |  |  |  |
| `additional_care` |  |  |  |

1. What does one row represent?
2. Which features were converted from numerical source codes to readable categorical labels before class?
3. Which feature records what happened after emergency-department treatment?


### Discussion notes

| Feature | Feature type | Data storage type | Missing values |
| --- | --- | --- | ---: |
| `treatment_date` | Temporal | `datetime64[ns]` | 0 |
| `age` | Quantitative | `int64` | 0 |
| `sex` | Categorical | `object` | 0 |
| `race` | Categorical | `object` | 0 |
| `body_part` | Categorical | `object` | 0 |
| `diagnosis` | Categorical | `object` | 0 |
| `disposition` | Categorical | `object` | 0 |
| `additional_care` | Categorical | `bool` | 0 |

1. One row represents one sampled injury case treated in a participating hospital emergency department.
2. `sex`, `race`, `body_part`, `diagnosis`, and `disposition` were converted from numerical codes to readable labels.
3. `disposition` records what happened after emergency-department treatment.


# Step 3 — Operation

The class dataset already contains the prepared categories needed by the dashboard. Our main operation is to summarize the cases that will appear in the interface.


In [ ]:
device_summary = injuries.groupby("device_type").agg(
    cases=("case_number", "size"),
    median_age=("age", "median"),
    additional_care_cases=("additional_care", "sum"),
)

device_summary["additional_care_percent"] = (
    100
    * device_summary["additional_care_cases"]
    / device_summary["cases"]
)


The original `disposition` feature contains categories such as treated and released, transferred, admitted, or held for observation. To keep one dashboard comparison manageable, the prepared `additional_care` feature groups cases that involved continued care after the emergency-department visit. The resulting percentage describes those outcomes in this dataset. It is not a clinical diagnosis of injury severity.


In [ ]:

all_device_order = [
    "Non-electric bicycle",
    "E-bike",
    "Unpowered scooter",
    "Powered scooter",
]

month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December",
]

all_devices = "All devices"

device_order = {
    all_devices: all_device_order,
    "Bicycles": ["Non-electric bicycle", "E-bike"],
    "Scooters": ["Unpowered scooter", "Powered scooter"],
    "Powered": ["E-bike", "Powered scooter"],
    "Unpowered": ["Non-electric bicycle", "Unpowered scooter"],
}

maximum_age_cases = injuries.groupby(["age", "device_type"]).size().max()
maximum_month_cases = injuries.groupby(["month", "device_type"]).size().max()
case_axis_maximum = max(maximum_age_cases, maximum_month_cases) * 1.1


# Step 4 — Check

Before building the interface, verify the observations and calculated features that will drive it.


In [ ]:
injuries[
    [
        "treatment_date",
        "device_type",
        "device_family",
        "age",
        "diagnosis",
        "body_part",
        "additional_care",
    ]
].isnull().sum()


In [ ]:
device_summary.round(1)


In [ ]:
injuries["treatment_date"].agg(["min", "max"])


## Discussion: check the calculated evidence

1. How many school-age cases are included in the prepared dataset?
2. Which device groups have relatively small numbers of cases, and how should that affect interpretation?
3. Why should these unweighted counts not be interpreted as national injury totals?
4. Why should the dashboard use `Powered scooter` rather than `E-scooter` as its label?


### Discussion notes

1. The dashboard uses 6,212 sampled cases involving patients ages 5–17.
2. The unpowered-scooter group has only 179 cases, and the non-electric bicycle group is much larger than the others. Patterns from small groups may be less stable.
3. The hospital sample was not collected so that raw cases occur in the same proportions as all U.S. injuries. National estimates require the survey-design fields omitted from this class dataset.
4. CPSC defines the relevant product code as powered scooters. The code does not identify the power source precisely enough to guarantee that every record involved an e-scooter.


# Step 5 — Evidence

## Build one static Plotly chart

Plotly Express, imported with the alias `px`, provides concise functions for creating interactive Plotly figures. A Dash graph displays an ordinary Plotly figure, so we create and inspect a figure before putting it inside the app.


In [ ]:
bicycle_data = injuries.loc[injuries["device_family"].eq("Bicycles")]

age_counts = (
    bicycle_data.groupby(["age", "device_type"], as_index=False)
    .size()
    .rename(columns={"size": "cases"})
)
age_counts.head()


Grouping by age and device type and then using `.size()` counts the records in each group.


In [ ]:
device_colors = {
    "Non-electric bicycle": "steelblue",
    "E-bike": "darkorange",
    "Unpowered scooter": "seagreen",
    "Powered scooter": "mediumpurple",
}

age_figure = px.line(
    age_counts,
    x="age",
    y="cases",
    color="device_type",
    markers=True,
    labels={
        "age": "Patient age (years)",
        "cases": "Cases in dataset",
        "device_type": "Device type",
    },
    title="Bicycle injury cases by patient age",
    color_discrete_map=device_colors,
)
age_figure.show()


Plotly accepts readable CSS color names as well as hexadecimal color codes. The dictionary assigns `steelblue`, `darkorange`, `seagreen`, and `mediumpurple` to the four device types. Reusing this dictionary gives each device the same color in every figure.


# Add an input: the dropdown

A dropdown uses a visible **label** and a value sent to Python. Its `value` sets the initial selection. `clearable=False` prevents an empty selection. This menu lets the user compare device families, compare powered with unpowered devices, or display all four devices together.


In [ ]:
family_dropdown = dcc.Dropdown(
    id="family-dropdown",
    options=[
        {"label": all_devices, "value": all_devices},
        {"label": "Bicycles", "value": "Bicycles"},
        {"label": "Scooters", "value": "Scooters"},
        {"label": "Powered", "value": "Powered"},
        {"label": "Unpowered", "value": "Unpowered"},
    ],
    value=all_devices,
    clearable=False,
)
family_dropdown


The `id` gives the component a unique name. Dash uses it to connect the dropdown to a callback.


# The smallest useful callback

Before updating several cards and figures, connect the dropdown to one sentence.


In [ ]:
callback_app = Dash(__name__)

callback_app.layout = html.Div(
    [
        html.H1("School-Age Micromobility Injury Dashboard"),
        family_dropdown,
        html.P(id="selection-message"),
    ]
)


The paragraph is initially empty. Its displayed content lives in its `children` property. The callback reads the dropdown's `value` and updates the paragraph's `children`.


In [ ]:
@callback_app.callback(
    Output("selection-message", "children"),
    Input("family-dropdown", "value"),
)
def show_selection(selected_family):
    return f"The dashboard is comparing: {selected_family}"


Run the app again and change the dropdown. This time the page responds to an input, even though it updates only one sentence.


In [ ]:
callback_app.run(jupyter_mode="tab", port=8052, debug=False)


```text
Input: family-dropdown.value
                ↓
Function: show_selection(selected_family)
                ↓
Output: selection-message.children
```

Dash calls the function whenever the dropdown value changes.


# Prepare reusable figure functions

The final callback must redraw its charts after filtering. The first function accepts the selected comparison's DataFrame and the requested horizontal axis. When the user chooses `All devices`, it draws four lines rather than two.


In [ ]:
def make_case_figure(filtered_injuries, selected_family, selected_x_axis):
    if selected_x_axis == "month":
        counts = (
            filtered_injuries.groupby(["device_type", "month"])
            .size()
            .unstack(fill_value=0)
            .reindex(columns=month_order, fill_value=0)
            .stack()
            .rename("cases")
            .reset_index()
        )
        x_feature = "month"
        x_label = "Month"
        title_phrase = "month"
        x_range = None
        category_orders = {
            "month": month_order,
            "device_type": device_order[selected_family],
        }
    else:
        counts = (
            filtered_injuries.groupby(["age", "device_type"], as_index=False)
            .size()
            .rename(columns={"size": "cases"})
        )
        x_feature = "age"
        x_label = "Patient age (years)"
        title_phrase = "patient age"
        x_range = [injuries["age"].min() - 0.5, injuries["age"].max() + 0.5]
        category_orders = {"device_type": device_order[selected_family]}

    return px.line(
        counts,
        x=x_feature,
        y="cases",
        color="device_type",
        markers=True,
        category_orders=category_orders,
        labels={
            x_feature: x_label,
            "cases": "Cases in dataset",
            "device_type": "Device type",
        },
        title=f"{selected_family} injury cases by {title_phrase}",
        color_discrete_map=device_colors,
        range_x=x_range,
        range_y=[0, case_axis_maximum],
    )


The function uses an `if` statement to decide how to group the records and which feature to place on the horizontal axis. For the monthly view, `.reindex(columns=month_order, fill_value=0)` puts the months in calendar order and supplies a zero if a device has no cases in a particular month. The same y-axis ceiling is calculated from the largest age-or-month count in the full dataset, so changing the device or axis selection does not visually rescale the comparison.


# Connect two controls to one chart

We keep the device-comparison dropdown and add a `dcc.RadioItems` control for the horizontal axis. The callback receives both values, selects the requested device records, and sends a new figure to the graph's `figure` property. The default view displays all four devices by patient age.


In [ ]:
case_chart_app = Dash(__name__)

case_chart_app.layout = html.Div(
    [
        html.H1("Injury Cases by Age or Month"),
        html.Label("Choose devices to compare:", htmlFor="case-device-dropdown"),
        dcc.Dropdown(
            id="case-device-dropdown",
            options=[
                {"label": all_devices, "value": all_devices},
                {"label": "Bicycles", "value": "Bicycles"},
                {"label": "Scooters", "value": "Scooters"},
                {"label": "Powered", "value": "Powered"},
                {"label": "Unpowered", "value": "Unpowered"},
            ],
            value=all_devices,
            clearable=False,
        ),
        html.Label("Choose the horizontal axis:", htmlFor="case-x-axis-toggle"),
        dcc.RadioItems(
            id="case-x-axis-toggle",
            options=[
                {"label": "Patient age", "value": "age"},
                {"label": "Month of year", "value": "month"},
            ],
            value="age",
            inline=True,
        ),
        dcc.Graph(id="case-comparison-graph"),
    ]
)


In [ ]:
@case_chart_app.callback(
    Output("case-comparison-graph", "figure"),
    Input("case-device-dropdown", "value"),
    Input("case-x-axis-toggle", "value"),
)
def update_case_chart(selected_devices, selected_x_axis):
    selected_injuries = injuries.loc[
        injuries["device_type"].isin(device_order[selected_devices])
    ]

    return make_case_figure(
        selected_injuries,
        selected_devices,
        selected_x_axis,
    )


Run the next cell to open this focused app in a browser tab. Try all five device options with both horizontal-axis settings. `All devices` should display four lines, while each other device option should display two.


In [ ]:
import webbrowser
from threading import Timer

case_chart_app_url = "http://127.0.0.1:8056"
Timer(1, webbrowser.open_new_tab, args=[case_chart_app_url]).start()
case_chart_app.run(jupyter_mode="external", port=8056, debug=False)


A port can be used by only one running app at a time. This focused app uses port 8056 because an earlier version of the lesson used port 8051. If Jupyter reports that port 8056 is already in use, stop the earlier app or restart the kernel before running this cell again.


The final chart compares a **percentage**, not a raw total. This helps us compare outcomes even though the device groups contain very different numbers of cases.


In [ ]:
def make_additional_care_figure(filtered_injuries, selected_family):
    outcomes = filtered_injuries.groupby("device_type", as_index=False).agg(
        cases=("case_number", "size"),
        additional_care_cases=("additional_care", "sum"),
    )
    outcomes["additional_care_percent"] = (
        100
        * outcomes["additional_care_cases"]
        / outcomes["cases"]
    )

    return px.bar(
        outcomes,
        x="device_type",
        y="additional_care_percent",
        color="device_type",
        category_orders={"device_type": device_order[selected_family]},
        labels={
            "device_type": "Device type",
            "additional_care_percent": "Cases in group (%)",
        },
        title="Share needing additional care",
        color_discrete_map=device_colors,
    )


In [ ]:
make_case_figure(bicycle_data, "Bicycles", "age")


In [ ]:
make_case_figure(bicycle_data, "Bicycles", "month")


In [ ]:
make_additional_care_figure(bicycle_data, "Bicycles")


# Add two filters for injury details

The family dropdown changes several high-level summaries. A different interaction can answer a more specific question:

> For all injuries or a selected diagnosis and body part, how many cases appear for each of the four device types?

The `diagnosis` feature supplies the type of injury, while `body_part` identifies the primary part of the body affected. We create one dropdown option for every value observed in each feature.


In [ ]:
diagnosis_counts = injuries["diagnosis"].value_counts()

all_diagnoses = "All injuries"

diagnosis_options = [{"label": all_diagnoses, "value": all_diagnoses}] + [
    {"label": diagnosis, "value": diagnosis}
    for diagnosis in sorted(diagnosis_counts[diagnosis_counts.gt(0)].index)
]

all_body_parts = "All body parts"

maximum_diagnosis_frequency = (
    injuries.groupby(["diagnosis", "device_type"])
    .size()
    .max()
)

diagnosis_axis_maximum = maximum_diagnosis_frequency * 1.1

maximum_all_injuries_frequency = (
    injuries.groupby("device_type")
    .size()
    .max()
)

all_injuries_axis_maximum = maximum_all_injuries_frequency * 1.1

body_part_options = [{"label": all_body_parts, "value": all_body_parts}] + [
    {
        "label": (
            "Entire body"
            if body_part == "All Parts Body"
            else body_part
        ),
        "value": body_part,
    }
    for body_part in sorted(injuries["body_part"].unique())
]


`diagnosis_counts` ensures the menu includes only diagnoses with at least one record in the prepared dataset. The added `All injuries` option tells the function not to apply a diagnosis filter and provides an overall summary across the four device types. Similarly, choosing `All body parts` tells the function not to apply a body-part filter.

The source data also contain an official category named `All Parts Body`. The dashboard displays that recorded injury category as `Entire body`. It is not the same as the dashboard instruction `All body parts`, which turns off the body-part filter.

The function filters by diagnosis, optionally filters by body part, counts the remaining records by device type, and uses `.reindex()` to keep all four device types in the chart even when one has zero matching cases.

Plotly normally adjusts an axis to fit the values in each new figure. That could make a bar representing one case fill as much vertical space as a bar representing many cases after the filters change. We calculate the largest possible bar in advance and use one consistent scale for all diagnosis-specific views. The `All injuries` overview needs a separate, larger scale because it includes every diagnosis. Multiplying each maximum by `1.1` leaves 10% of the scale empty above its largest bar.


In [ ]:
def make_injury_frequency_figure(selected_diagnosis, selected_body_part):
    selected_cases = injuries

    if selected_diagnosis != all_diagnoses:
        selected_cases = selected_cases.loc[
            selected_cases["diagnosis"].eq(selected_diagnosis)
        ]

    if selected_body_part != all_body_parts:
        selected_cases = selected_cases.loc[
            selected_cases["body_part"].eq(selected_body_part)
        ]

    counts = (
        selected_cases.groupby("device_type")
        .size()
        .reindex(all_device_order, fill_value=0)
        .rename("cases")
        .reset_index()
    )

    body_part_title = (
        "all body parts"
        if selected_body_part == all_body_parts
        else (
            "the entire body"
            if selected_body_part == "All Parts Body"
            else f"the {selected_body_part.lower()}"
        )
    )

    axis_maximum = (
        all_injuries_axis_maximum
        if selected_diagnosis == all_diagnoses
        else diagnosis_axis_maximum
    )

    return px.bar(
        counts,
        x="device_type",
        y="cases",
        color="device_type",
        text="cases",
        range_y=[0, axis_maximum],
        category_orders={"device_type": all_device_order},
        labels={
            "device_type": "Device type",
            "cases": "Cases in dataset",
        },
        title=f"{selected_diagnosis} affecting {body_part_title}",
        color_discrete_map=device_colors,
    )


`text="cases"` prints the exact frequency on each bar. The labels remain useful when a small bar is difficult to see on the fixed scale.


Before assembling the full page, run a focused version containing only the two controls and their graph.


In [ ]:
injury_explorer_app = Dash(__name__)

injury_explorer_app.layout = html.Div(
    [
        html.H1("Explore injury details"),
        html.Label("Choose a diagnosis:", htmlFor="diagnosis-dropdown"),
        dcc.Dropdown(
            id="diagnosis-dropdown",
            options=diagnosis_options,
            value=all_diagnoses,
            clearable=False,
        ),
        html.Label("Choose a body part:", htmlFor="body-part-dropdown"),
        dcc.Dropdown(
            id="body-part-dropdown",
            options=body_part_options,
            value=all_body_parts,
            clearable=False,
        ),
        dcc.Graph(id="injury-frequency-graph"),
    ]
)


In [ ]:
@injury_explorer_app.callback(
    Output("injury-frequency-graph", "figure"),
    Input("diagnosis-dropdown", "value"),
    Input("body-part-dropdown", "value"),
)
def update_injury_frequency(selected_diagnosis, selected_body_part):
    return make_injury_frequency_figure(
        selected_diagnosis,
        selected_body_part,
    )


This callback has two inputs. Changing either dropdown causes Dash to run the function again with both current selections.


In [ ]:
injury_explorer_app.run(jupyter_mode="tab", port=8053, debug=False)


## Discussion: test the injury filters

1. Begin with `All injuries` and `All body parts`. What overall pattern do you see across the four device types?
2. Choose a specific diagnosis. What changes in the graph?
3. Change only the body part. Why is it useful for all four device types to remain visible even when a count is zero?


# Assemble the final dashboard

We now have the individual pieces. The final reading order is title, controls, summary cards, overview charts, injury-detail controls, and source note.


In [ ]:
card_style = {
    "backgroundColor": "white",
    "border": "1px solid #D9E2E3",
    "borderRadius": "8px",
    "padding": "16px",
    "flex": "1",
    "minWidth": "180px",
}

dashboard_app = Dash(__name__)


`flex` lets cards share a row, while `minWidth` keeps them readable. `flexWrap="wrap"` in the parent lets items move to another row on smaller screens.


In [ ]:
dashboard_app.layout = html.Div(
    [
        html.H1("School-Age Micromobility Injury Dashboard"),
        html.P(
            "Compare 2025 emergency-department cases for patients ages 5–17 in the class dataset."
        ),
        html.Label("Choose devices to compare:", htmlFor="final-family-dropdown"),
        dcc.Dropdown(
            id="final-family-dropdown",
            options=[
                {"label": all_devices, "value": all_devices},
                {"label": "Bicycles", "value": "Bicycles"},
                {"label": "Scooters", "value": "Scooters"},
                {"label": "Powered", "value": "Powered"},
                {"label": "Unpowered", "value": "Unpowered"},
            ],
            value=all_devices,
            clearable=False,
        ),
        html.Label("Choose the horizontal axis:", htmlFor="final-x-axis-toggle"),
        dcc.RadioItems(
            id="final-x-axis-toggle",
            options=[
                {"label": "Patient age", "value": "age"},
                {"label": "Month of year", "value": "month"},
            ],
            value="age",
            inline=True,
        ),
        html.Div(
            [
                html.Div([html.P("Cases in dataset"), html.H2(id="case-count")], style=card_style),
                html.Div([html.P("Median patient age"), html.H2(id="median-age")], style=card_style),
                html.Div(
                    [
                        html.P("Needed additional care"),
                        html.H2(id="additional-care-share"),
                    ],
                    style=card_style,
                ),
            ],
            style={"display": "flex", "gap": "12px", "flexWrap": "wrap", "marginTop": "20px"},
        ),
        dcc.Graph(id="case-graph"),
        dcc.Graph(id="additional-care-graph"),
        html.H2("Explore injury details"),
        html.P(
            "Choose a diagnosis and primary body part to compare matching cases across all four device types."
        ),
        html.Div(
            [
                html.Div(
                    [
                        html.Label("Choose a diagnosis:", htmlFor="final-diagnosis-dropdown"),
                        dcc.Dropdown(
                            id="final-diagnosis-dropdown",
                            options=diagnosis_options,
                            value=all_diagnoses,
                            clearable=False,
                        ),
                    ],
                    style={"flex": "1", "minWidth": "280px"},
                ),
                html.Div(
                    [
                        html.Label("Choose a body part:", htmlFor="final-body-part-dropdown"),
                        dcc.Dropdown(
                            id="final-body-part-dropdown",
                            options=body_part_options,
                            value=all_body_parts,
                            clearable=False,
                        ),
                    ],
                    style={"flex": "1", "minWidth": "280px"},
                ),
            ],
            style={"display": "flex", "gap": "12px", "flexWrap": "wrap"},
        ),
        dcc.Graph(id="final-injury-frequency-graph"),
        html.P(
            "Source: U.S. Consumer Product Safety Commission, National Electronic Injury "
            "Surveillance System (NEISS), 2025 query for product codes 5022, 5023, 5033, "
            "5040, and 5045. The dashboard shows unweighted cases in the prepared class "
            "dataset, not national estimates. CPSC code 5022 identifies powered scooters, "
            "not exclusively e-scooters.",
            style={"color": "#4B5B5D", "fontSize": "0.9rem"},
        ),
    ],
    style={
        "fontFamily": "Arial, sans-serif",
        "maxWidth": "1200px",
        "margin": "0 auto",
        "padding": "24px",
        "backgroundColor": "#F5F8F8",
    },
)


# Connect the final callbacks

The main callback follows the same pattern as the small callback, but it receives two inputs and updates five outputs. Returned values must match the outputs in both **number** and **order**.


In [ ]:
@dashboard_app.callback(
    Output("case-count", "children"),
    Output("median-age", "children"),
    Output("additional-care-share", "children"),
    Output("case-graph", "figure"),
    Output("additional-care-graph", "figure"),
    Input("final-family-dropdown", "value"),
    Input("final-x-axis-toggle", "value"),
)
def update_dashboard(selected_family, selected_x_axis):
    filtered_injuries = injuries.loc[
        injuries["device_type"].isin(device_order[selected_family])
    ]

    case_count = f"{len(filtered_injuries):,}"
    median_age = f"{filtered_injuries['age'].median():.0f} years"
    additional_care_share = (
        f"{100 * filtered_injuries['additional_care'].mean():.1f}%"
    )

    return (
        case_count,
        median_age,
        additional_care_share,
        make_case_figure(
            filtered_injuries,
            selected_family,
            selected_x_axis,
        ),
        make_additional_care_figure(filtered_injuries, selected_family),
    )


The injury-detail controls use a second callback. Keeping this interaction separate means changing a diagnosis or body part redraws only the injury-frequency graph.


In [ ]:
@dashboard_app.callback(
    Output("final-injury-frequency-graph", "figure"),
    Input("final-diagnosis-dropdown", "value"),
    Input("final-body-part-dropdown", "value"),
)
def update_final_injury_frequency(selected_diagnosis, selected_body_part):
    return make_injury_frequency_figure(
        selected_diagnosis,
        selected_body_part,
    )


Trace one change:

1. The user chooses `Scooters` and `Month of year`.
2. Dash passes `"Scooters"` to `selected_family` and `"month"` to `selected_x_axis`.
3. The function selects the two scooter groups.
4. It calculates three summaries and creates two figures, using month for the first figure.
5. Dash places the five returned values in the five output properties.


## Check the callback

A callback is still a Python function, so we can call it directly.


In [ ]:
test_result = update_dashboard("Scooters", "month")

print("Cases in dataset:", test_result[0])
print("Median age:", test_result[1])
print("Needed additional care:", test_result[2])
print("Number of returned items:", len(test_result))


## Run the completed dashboard


In [ ]:
dashboard_app.run(jupyter_mode="tab", port=8054, debug=False)


## Discussion: interpret the dashboard evidence

1. Use the dropdown to view the bicycle, scooter, powered, and unpowered comparisons. Which device has more cases within each pair?
2. When all four age lines are displayed, which similarities and differences are easiest to see?
3. Switch the horizontal axis to month. When are case counts highest, and does the monthly pattern differ among devices?
4. Within each family, which device has the larger share needing additional care?
5. Why is a percentage more useful than a raw count for comparing these outcomes?
6. Select one diagnosis and body part. How do the matching case counts differ across the four devices?
7. Which result could motivate a focused follow-up study?


### Discussion notes

Non-electric bicycles have more cases than e-bikes, while powered scooters have more than unpowered scooters. E-bike and powered-scooter cases are concentrated among somewhat older children than their comparison groups. Monthly case counts are generally higher during warmer months, although the strength and timing of the pattern differ among devices. In the unweighted data, approximately 14.6% of e-bike cases needed additional care versus 6.6% of non-electric bicycle cases. The corresponding shares are approximately 11.0% for powered scooters and 7.3% for unpowered scooters. Percentages make the outcome comparison less dependent on the very different numbers of cases in each group. Answers for the diagnosis and body-part comparison depend on the selected categories.


# Step 6 — Conclusion and limitation

In this prepared 2025 dataset of patients ages 5–17, non-electric bicycles have the most cases, but e-bikes have a larger share of cases needing additional care. Powered scooters have both more cases and a larger share needing additional care than unpowered scooters. The diagnosis and body-part controls show that the comparison can change for particular kinds of injuries. These patterns support continued investigation of powered micromobility safety among young riders, but they do not show that electric power caused an injury or predict the effect of the OCPS school-board decision.

## Limitations

- NEISS covers injuries treated in hospital emergency departments, not every micromobility injury.
- The dashboard uses unweighted records. This is not a representative sample that can be used to make national estimates.
- The data are national, not specific to Orange County or students traveling to school.
- Patient age does not establish whether a child was a student or commuting to school.
- The data describe 2025, before the 2026–2027 OCPS decision takes effect.
- CPSC code 5022 includes powered scooters generally and does not identify every case as an e-scooter.
- The dataset does not measure how often or how far each type of device was ridden, so it cannot estimate injury risk per trip or mile.
- Ninety-seven records containing more than one dashboard group were excluded to make the groups mutually exclusive.
- Product association does not by itself establish that a product defect or electric motor caused an injury.


# Final design review

- Can the intended audience tell what question the dashboard answers?
- Is the default view informative?
- Does every element earn its space?
- Are labels, units, colors, and reading order clear?
- Does each device type use a consistent color?
- Does every interaction produce a meaningful comparison?
- Is it clear that every frequency is a count of records in this dataset?
- Are calculated categories defined near the evidence?
- Are source, date, population, and limitations visible?


## Discussion: review the dashboard

1. What is one responsible action a school-safety researcher could take after seeing this dashboard?
2. What additional evidence would be needed to evaluate the OCPS school-board decision?
3. If the dashboard had room for only one chart, which would you keep and why?
4. How does the dashboard continue the analysis workflow rather than replace it?


### Discussion notes

1. A responsible next step would be to define a focused question around age, month, injury outcome, diagnosis, or body part and gather local crash, hospital, and device-use information.
2. Evaluating the decision would require comparable Orange County data before and after implementation, information about ridership and implementation, and attention to other changes occurring over time.
3. Answers will vary. Students should justify the choice using the dashboard's question and audience rather than personal chart preference alone.
4. The dashboard presents operations and evidence selected through the workflow. The underlying question, data checks, interpretation, and limitations remain necessary.


# Summary

A useful dashboard is organized around a specific audience and question. Plotly Express creates the figures, while Dash arranges components in a layout and uses callbacks to connect user inputs to updated outputs. The micromobility example also shows why fair comparisons require percentages, consistent categories, careful labels, and clear limitations. Multiple callbacks can support separate interactions without making every part of the page update at once. The interface makes checked evidence easier to explore, but it does not make the evidence stronger or replace the analysis workflow.
